# 02 — Data Cleaning & Feature Engineering
Transforms raw match data into a model-ready dataset: results, points, gameweek, title gap, high-stakes flags, Drop Index, rolling features, and recency weights.

## Imports

In [1]:
import pandas as pd
import numpy as np
import soccerdata as sd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print("\n✅ Imports ready")

[08/27/26 00:54:40] INFO     No custom team name replacements found. You can configure these in       ]8;id=7056935;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=7056936;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\tejas\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=7056942;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=7056943;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\tejas\soccerdata\config\league_dict.json.                                    

pandas : 3.0.5
numpy  : 2.4.6

✅ Imports ready


## Paths & Config

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROC_DATA_DIR.mkdir(parents=True, exist_ok = True)

TITLE_TEAMS = ['Arsenal','Liverpool','Manchester City','Manchester United']
PL = "ENG-Premier League"
ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

# Recency weights per season — 25-26 counts 1.5x in the ML model
SEASON_WEIGHTS = {
    '1920': 1.0, '2021': 1.0, '2122': 1.0,
    '2223': 1.1, '2324': 1.2, '2425': 1.3,
    '2526': 1.5,
}

print(f"Raw data dir       : {RAW_DATA_DIR}")
print(f"Processed data dir : {PROC_DATA_DIR}")
print(f"Season weights     : {SEASON_WEIGHTS}")

Raw data dir       : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\raw
Processed data dir : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\processed
Season weights     : {'1920': 1.0, '2021': 1.0, '2122': 1.0, '2223': 1.1, '2324': 1.2, '2425': 1.3, '2526': 1.5}


## Load and Combine All 4 Teams

In [3]:
# Load all 4 title teams and stack into one DataFrame

dfs = [
    pd.read_csv(RAW_DATA_DIR / f"{team.lower().replace(' ', '_')}_raw.csv")
    for team in TITLE_TEAMS
]

df = pd.concat(dfs, ignore_index = True)

# Fix date column type
df['date'] = pd.to_datetime(df['date'])

# Sort by team then date — critical for rolling features to work correctly
df = df.sort_values(['team','date']).reset_index(drop=True)


print(f"Combined shape : {df.shape}")
print(f"Teams          : {sorted(df['team'].unique())}")
print(f"Date range     : {df['date'].min().date()} → {df['date'].max().date()}")
df.head(5)

Combined shape : (1064, 11)
Teams          : ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
Date range     : 2019-08-09 → 2026-05-24


,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded
0,ENG-Premier League,1920,11650,2019-08-11 14:00:00,Arsenal,Newcastle United,away,1.133090,0.380551,1,0
1,ENG-Premier League,1920,11653,2019-08-17 12:30:00,Arsenal,Burnley,home,1.164400,1.391720,2,1
2,ENG-Premier League,1920,11670,2019-08-24 17:30:00,Arsenal,Liverpool,away,0.985542,2.788210,1,3
3,ENG-Premier League,1920,11682,2019-09-01 16:30:00,Arsenal,Tottenham,home,1.925090,1.955140,2,2
4,ENG-Premier League,1920,11691,2019-09-15 15:30:00,Arsenal,Watford,away,1.006160,2.832090,2,2


## Compute Result and Points

In [4]:
# Compute result — W/D/L from scored vs conceded

df['result'] = np.where(
    df['scored'] > df['conceded'], 'W',
    np.where(df['scored'] == df['conceded'], 'D', 'L')
)

# Compute points — 3 for win, 1 for draw, 0 for loss
df['points'] = df['result'].map({'W' : 3, 'D' : 1, 'L' : 0})

# Compute goal difference per match
df['gd'] = df['scored'] - df['conceded']

# Compute xG difference
df['xgd'] = df['xG'] - df['xGA']

print("Result distribution across all 4 teams:")
print(df['result'].value_counts())
print()
print("Points distribution:")
print(df['points'].value_counts().sort_index())
print()
print("Spot check — first 5 Arsenal rows:")
df[df['team'] == 'Arsenal'][['date', 'opponent', 'scored', 'conceded', 'result', 'points']].head(5)

Result distribution across all 4 teams:
result
W    628
D    222
L    214
Name: count, dtype: int64

Points distribution:
points
0    214
1    222
3    628
Name: count, dtype: int64

Spot check — first 5 Arsenal rows:


,date,opponent,scored,conceded,result,points
0,2019-08-11 14:00:00,Newcastle United,1,0,W,3
1,2019-08-17 12:30:00,Burnley,2,1,W,3
2,2019-08-24 17:30:00,Liverpool,1,3,L,0
3,2019-09-01 16:30:00,Tottenham,2,2,D,1
4,2019-09-15 15:30:00,Watford,2,2,D,1


## Add Gameweek

In [5]:
# Compute gameweek: rank matches chronologically within each team-season

df['gameweek'] = (
    df.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

# Verify — each team in each season should have gameweeks 1 to 38
gw_check = df.groupby(['team', 'season'])['gameweek'].max().unstack(fill_value=0)
print("Max gameweek per team per season (should all be 38):")
print(gw_check)

Max gameweek per team per season (should all be 38):
season             1920  2021  2122  2223  2324  2425  2526
team                                                       
Arsenal              38    38    38    38    38    38    38
Liverpool            38    38    38    38    38    38    38
Manchester City      38    38    38    38    38    38    38
Manchester United    38    38    38    38    38    38    38


## Reconstruct Full PL Title Table

In [6]:
print("Loading full 20-team schedule from cache...")

understat = sd.Understat(leagues=PL, seasons=ARTETA_SEASONS)
full_schedule = understat.read_schedule().reset_index()

print(f"Shape: {full_schedule.shape}")
print("Done — loaded from local cache")

Loading full 20-team schedule from cache...


                    INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=7056950;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=7056951;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-08-27 00:54:40] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=7056958;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=7056959;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\                 
                             bin\tls-client-xgo-1.13.1-windows-amd64.dll                                           

Shape: (2660, 20)
Done — loaded from local cache


In [7]:
KEEP = ['league', 'season', 'game_id', 'date', 'team', 'scored', 'conceded']

full_home = full_schedule.rename(columns={
    'home_team': 'team', 'home_goals': 'scored', 'away_goals': 'conceded'
})
full_away = full_schedule.rename(columns={
    'away_team': 'team', 'away_goals': 'scored', 'home_goals': 'conceded'
})

full_long = pd.concat([full_home[KEEP],full_away[KEEP]],ignore_index=True)
full_long['date'] = pd.to_datetime(full_long['date'])

# Compute points for each team
full_long['points'] = np.where(
    full_long['scored'] > full_long['conceded'],3,
    np.where(full_long['scored'] == full_long['conceded'],1,0)
)

# Sort chronologically within each team-season before cumsum
full_long = full_long.sort_values(['team','season','date']).reset_index(drop = True)

print(f"Full 20-team long format: {full_long.shape}")


Full 20-team long format: (5320, 8)


In [8]:
#Compute cumsum

full_long['cum_pts'] = full_long.groupby(['team','season'])['points'].cumsum()

full_long['gw'] = (
    full_long.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

print("Arsenal cumulative points (first 5 GW of 2019-20):")
mask = (full_long['team'] == 'Arsenal') & (full_long['season'] == '1920')
print(full_long[mask][['date', 'gw', 'points', 'cum_pts']].head(5))

Arsenal cumulative points (first 5 GW of 2019-20):
                 date  gw  points  cum_pts
0 2019-08-11 14:00:00   1       3        3
1 2019-08-17 12:30:00   2       3        6
2 2019-08-24 17:30:00   3       0        6
3 2019-09-01 16:30:00   4       1        7
4 2019-09-15 15:30:00   5       1        8


## Compute Title Gap

In [9]:
# Each Gameweek's leader's points

full_long['leader_pts'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform('max')
)

#Title Gap = Leader's points - the team's points

full_long['title_gap'] = full_long['leader_pts'] - full_long['cum_pts']

#Quick check
mask = (full_long['season'] == '1920') & (full_long['gw'] == 30)
print("GW 30 2019-20 - points and title gap: ")
print(
    full_long[mask][['team','cum_pts','leader_pts','title_gap']]
    .sort_values('title_gap').head(10)
)

# Extracting title_gap for only our 4 temas

title_gap_df = (
    full_long[full_long['team'].isin(TITLE_TEAMS)]
    [['season','team','gw','title_gap','cum_pts','leader_pts']]
    .copy()
    .rename(columns={'gw':'gameweek'})
)

df['season'] = df['season'].astype('str')
print(f"Title gap table shape: {title_gap_df.shape}")
print(f"Expected: 4 teams × 7 seasons × 38 GW = {4*7*38} rows")
title_gap_df.head(5)

GW 30 2019-20 - points and title gap: 
                         team  cum_pts  leader_pts  title_gap
2765                Liverpool       83          83          0
3069          Manchester City       63          83         20
2575                Leicester       54          83         29
1397                  Chelsea       51          83         32
3335        Manchester United       46          83         37
5083  Wolverhampton Wanderers       46          83         37
4095         Sheffield United       44          83         39
1663           Crystal Palace       42          83         41
4437                Tottenham       42          83         41
29                    Arsenal       40          83         43
Title gap table shape: (1064, 6)
Expected: 4 teams × 7 seasons × 38 GW = 1064 rows


,season,team,gameweek,title_gap,cum_pts,leader_pts
0,1920,Arsenal,1,0,3,3
1,1920,Arsenal,2,0,6,6
2,1920,Arsenal,3,3,6,9
3,1920,Arsenal,4,5,7,12
4,1920,Arsenal,5,7,8,15


## Merge Title Gap into Main DataFrame

In [10]:
# Merge title_gap into our main df

df = pd.merge(
    df,
    title_gap_df,
    on=['season', 'team', 'gameweek'],
    how='left'
)


print(f"Shape after merge: {df.shape}")
print(f"title_gap nulls  : {df['title_gap'].isna().sum()}")

# Spot check — Arsenal 2021-22, final gameweek
mask = (df['team'] == 'Arsenal') & (df['season'] == '2122') & (df['gameweek'] >= 36)
df[mask][['date', 'gameweek', 'opponent', 'result', 'cum_pts', 'title_gap']].sort_values('gameweek')

Shape after merge: (1064, 19)
title_gap nulls  : 0


,date,gameweek,opponent,result,cum_pts,title_gap
111,2022-05-12 18:45:00,36,Tottenham,L,66,23
112,2022-05-16 19:00:00,37,Newcastle United,L,66,24
113,2022-05-22 15:00:00,38,Everton,W,69,24


In [11]:
# Check what your raw data actually has for those last 3 Arsenal 2021-22 matches
mask = (df['team'] == 'Arsenal') & (df['season'] == '2122') & (df['gameweek'] >= 36)
print("From your df (raw CSV source):")
print(df[mask][['date', 'gameweek', 'opponent', 'venue',
                 'scored', 'conceded', 'result', 'points']].to_string())
print()
# Now check what full_long has for the same matches
fl_mask = (full_long['team'] == 'Arsenal') & (full_long['season'] == '2122') & (full_long['gw'] >= 36)
print("From full_long (fresh Understat pull):")
print(full_long[fl_mask][['date', 'gw', 'scored', 'conceded', 'points', 'cum_pts']].to_string())

From your df (raw CSV source):
                   date  gameweek          opponent venue  scored  conceded result  points
111 2022-05-12 18:45:00        36         Tottenham  away       0         3      L       0
112 2022-05-16 19:00:00        37  Newcastle United  away       0         2      L       0
113 2022-05-22 15:00:00        38           Everton  home       5         1      W       3

From full_long (fresh Understat pull):
                   date  gw  scored  conceded  points  cum_pts
111 2022-05-12 18:45:00  36       0         3       0       66
112 2022-05-16 19:00:00  37       0         2       0       66
113 2022-05-22 15:00:00  38       5         1       3       69


## Define High-Stakes Matches

In [12]:
# Component 1: time pressure — sigmoid accelerating past GW22
gw_factor = 1 / (1 + np.exp(-0.2 * (df['gameweek'] - 22)))

# Component 2: competitive closeness — linear decay from gap=0 to gap=15
gap_factor = (1 - df['title_gap'] / 15).clip(lower=0)

# Combined: multiply the two components
df['stakes_intensity'] = (gw_factor * gap_factor).round(4)

print("stakes_intensity distribution: ")
print(df['stakes_intensity'].describe().round(3))
print()


print("Mean stakes_intensity per team: ")
print(df.groupby('team')['stakes_intensity'].mean().round(3))
print()

ars = df[df['team'] == 'Arsenal'].sort_values('stakes_intensity',ascending=False)
print("Arsenal top 5 highest-stakes matches:")
print(ars[['date', 'season', 'gameweek', 'opponent', 'title_gap', 'stakes_intensity']].head(5).to_string())

stakes_intensity distribution: 
count    1064.000
mean        0.193
std         0.282
min         0.000
25%         0.000
50%         0.045
75%         0.256
max         0.961
Name: stakes_intensity, dtype: float64

Mean stakes_intensity per team: 
team
Arsenal              0.195
Liverpool            0.243
Manchester City      0.284
Manchester United    0.051
Name: stakes_intensity, dtype: float64

Arsenal top 5 highest-stakes matches:
                   date season  gameweek          opponent  title_gap  stakes_intensity
265 2026-05-24 15:00:00   2526        38    Crystal Palace          0            0.9608
264 2026-05-18 20:00:00   2526        37           Burnley          0            0.9526
263 2026-05-10 16:30:00   2526        36          West Ham          0            0.9427
262 2026-05-02 17:30:00   2526        35            Fulham          0            0.9309
261 2026-04-25 16:30:00   2526        34  Newcastle United          0            0.9168


## Build the Drop Index

In [13]:
df.sort_values('stakes_intensity',ascending = False)

,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded,result,points,gd,xgd,gameweek,title_gap,cum_pts,leader_pts,stakes_intensity
645,ENG-Premier League,2122,16754,2022-05-22 15:00:00,Manchester City,Aston Villa,home,3.320900,0.247841,3,2,W,3,1,3.073059,38,0,93,93,0.9608
683,ENG-Premier League,2223,18574,2023-05-28 15:30:00,Manchester City,Brentford,away,1.186490,1.438930,0,1,L,0,-1,-0.252440,38,0,89,89,0.9608
721,ENG-Premier League,2324,22273,2024-05-19 15:00:00,Manchester City,West Ham,home,2.315040,0.240159,3,1,W,3,2,2.074881,38,0,91,91,0.9608
607,ENG-Premier League,2021,14811,2021-05-23 15:00:00,Manchester City,Everton,home,2.883550,1.072580,5,0,W,3,5,1.810970,38,0,86,86,0.9608
493,ENG-Premier League,2425,26975,2025-05-25 15:00:00,Liverpool,Crystal Palace,home,1.921220,1.414400,1,1,D,1,0,0.506820,38,0,84,84,0.9608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1011,ENG-Premier League,2425,26838,2025-02-02 14:00:00,Manchester United,Crystal Palace,home,1.507060,2.446330,0,2,L,0,-2,-0.939270,24,28,29,57,0.0000
1012,ENG-Premier League,2425,26850,2025-02-16 16:30:00,Manchester United,Tottenham,away,1.426520,1.971090,0,1,L,0,-1,-0.544570,25,31,29,60,0.0000
1013,ENG-Premier League,2425,26855,2025-02-22 12:30:00,Manchester United,Everton,away,0.645685,2.104440,2,2,D,1,0,-1.458755,26,31,30,61,0.0000
1014,ENG-Premier League,2425,26871,2025-02-26 19:30:00,Manchester United,Ipswich,home,1.402020,1.044980,3,2,W,3,1,0.357040,27,31,33,64,0.0000


## Rolling Features
**Critical:** always `.shift(1)` before `.rolling()` — no exceptions.

## Recency Weights

## Sanity Checks

## Save to Processed